# The Resilience Test: What Did Diversification Actually Protect?

## Executive question
For a hypothetical investor choosing from the stocks in this historical file, did diversification reduce losses reliably—or did the apparent benefit depend on portfolio size, rebalance frequency, and the market window?

This is a **historical portfolio resilience audit**, not a stock-picking exercise. It compares concentration and equal-weight diversification, measures downside losses and drawdowns, tests results across rolling windows, and challenges conclusions with sensitivity checks.

**Dataset:** `all_stocks_5yr.csv`, observed in this upload as 619,040 rows and 505 tickers, daily data from 2013-02-08 through 2018-02-07. This analysis cannot establish current investment opportunities. Corporate-action adjustment and constituent-survivorship status are unverified; those are material limitations, not footnotes.

### Research hypotheses
1. Increasing the number of holdings reduces portfolio volatility and tail loss, but has diminishing marginal benefit.
2. Diversification may reduce drawdown severity without improving returns.
3. The measured benefit may vary across rolling windows and rebalance assumptions.

### Evidence standard
All portfolios use only tickers with a complete common daily return panel after a minimum-history screen. Holdings are selected using training-period liquidity only, then evaluated on later data. No future return is used to select a portfolio. This improves the comparison but does not remove survivorship bias in the source universe.

## 1. Reproducible setup
Set `DATA_PATH` to the CSV's location. Outputs and figures are written to `images/`. The notebook uses only widely available data-analysis libraries.

In [2]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 6)
DATA_PATH = Path('all_stocks_5yr.csv')  # edit if needed
OUT = Path('images'); OUT.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.head()

ModuleNotFoundError: No module named 'numpy'

## 2. Data audit: what can this dataset support?
Before analyzing returns, verify schema, date coverage, missingness, duplicate ticker-dates, price validity, and ticker coverage. Missing OHLC fields are not imputed; close-to-close returns require valid closes. If the source documentation confirms adjusted prices, document that here. Otherwise interpret results as price-return analysis only.

In [ ]:
expected = {'date','open','high','low','close','volume','Name'}
assert expected.issubset(df.columns), f'Missing columns: {expected-set(df.columns)}'
df['Name'] = df['Name'].astype(str).str.strip()
for c in ['open','high','low','close','volume']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.drop_duplicates(['Name','date'], keep='last').sort_values(['Name','date']).copy()
print('Rows / columns:', df.shape, '| tickers:', df.Name.nunique())
print('Date range:', df.date.min().date(), 'to', df.date.max().date())
display(df.isna().sum().rename('missing').to_frame())
print('Invalid nonpositive closes:', (df.close <= 0).sum())
coverage = df.groupby('Name').agg(n=('date','nunique'), first=('date','min'), last=('date','max'))
display(coverage.n.describe(percentiles=[.1,.25,.5,.75,.9]).to_frame('observations_per_ticker'))

### Coverage visualization
A strategy based on a ticker universe with uneven histories can accidentally compare different eras. The next chart shows how much each ticker contributes; later, we form a common-panel investable subset using training-only information.

In [ ]:
coverage.n.sort_values().reset_index(drop=True).plot(color='#2878b5', title='Daily observations available per ticker')
plt.xlabel('Ticker rank by history length'); plt.ylabel('Observed dates'); plt.tight_layout()
plt.savefig(OUT/'01_ticker_coverage.png', dpi=160); plt.show()

## 3. Return engineering and time split
We calculate close-to-close returns by ticker, then divide dates chronologically: first 60% for universe selection and calibration, remaining 40% for evaluation. Tickers must have at least 90% of expected observations in each segment to enter the common panel. This rule is explicit and reproducible, but the original file may already have survivorship bias.

In [ ]:
df = df.dropna(subset=['date','Name','close','volume'])
df = df[df.close > 0].copy()
df['ret'] = df.groupby('Name').close.pct_change()
df['dollar_volume'] = df.close * df.volume
dates = np.array(sorted(df.date.unique()))
cut = dates[int(len(dates)*0.60)]
print('Training ends:', pd.Timestamp(cut).date(), '| test starts:', pd.Timestamp(cut).date(), '(boundary date shared only for screening)')
train = df[df.date < cut].copy(); test = df[df.date >= cut].copy()
expected_train = train.date.nunique(); expected_test = test.date.nunique()
train_cov = train.groupby('Name').ret.count()/expected_train
test_cov = test.groupby('Name').ret.count()/expected_test
eligible = train_cov[(train_cov >= .90) & (test_cov.reindex(train_cov.index).fillna(0) >= .90)].index
print('Tickers passing common-panel coverage:', len(eligible))
if len(eligible) < 20: raise ValueError('Too few tickers pass coverage screen; inspect coverage and adjust documented threshold.')
train_panel = train[train.Name.isin(eligible)].pivot(index='date', columns='Name', values='ret').sort_index()
test_panel = test[test.Name.isin(eligible)].pivot(index='date', columns='Name', values='ret').sort_index()
# Keep only tickers and dates with complete returns; first return per ticker is naturally missing.
common = train_panel.columns.intersection(test_panel.columns)
train_panel = train_panel[common].dropna(axis=0, how='any')
test_panel = test_panel[common].dropna(axis=0, how='any')
print('Common tickers:', len(common), '| complete train dates:', len(train_panel), '| complete test dates:', len(test_panel))
if len(test_panel) < 100: raise ValueError('Test panel too short after complete-case filtering; review screen.')

## 4. Training-only portfolio selection
To avoid choosing historical winners, portfolio candidates are ranked by average daily dollar volume during training only. We compare the same liquid candidate pool at different breadths. Equal weights are reset monthly using the available daily panel; this is a transparent return-stream approximation, not a broker-executable backtest. We also compare with a training-period minimum-volatility weighting as a sensitivity, with weights capped to reduce concentration.

In [ ]:
liq = train[train.Name.isin(common)].groupby('Name').dollar_volume.mean().reindex(common).sort_values(ascending=False)
candidate_order = liq.index.tolist()
print('Top training-period liquid candidates:', candidate_order[:10])
def monthly_equal_weight(panel, names):
    r = panel[names].copy()
    # Equal weights reset at the first trading date of each month; daily buy-and-hold within month.
    out = pd.Series(index=r.index, dtype=float)
    month_key = r.index.to_period('M')
    for m in month_key.unique():
        idx = month_key == m
        block = r.loc[idx]
        out.loc[block.index] = block.mean(axis=1)
    return out.dropna()
def minvol_weights(panel, names, cap=0.10):
    vol = panel[names].std().replace(0, np.nan).dropna()
    w = (1/vol); w = w/w.sum()
    # Iterative cap-and-redistribute, then normalize (approximate capped inverse-vol weights).
    for _ in range(20):
        over = w > cap
        if not over.any(): break
        excess = (w[over]-cap).sum(); w[over] = cap
        under = ~over
        if under.any(): w[under] += excess*w[under]/w[under].sum()
    return w/w.sum()
def weighted_returns(panel, weights):
    return panel[weights.index].mul(weights, axis=1).sum(axis=1)
breadths = [5, 10, 20, 40]
breadths = [n for n in breadths if n <= len(candidate_order)]
test_returns = {}
for n in breadths:
    names = candidate_order[:n]
    test_returns[f'Equal weight {n}'] = monthly_equal_weight(test_panel, names)
n_full = min(40, len(candidate_order))
w_iv = minvol_weights(train_panel, candidate_order[:n_full], cap=.10)
test_returns[f'Capped inverse-vol {n_full}'] = weighted_returns(test_panel, w_iv)
print('Weights sum:', round(w_iv.sum(),6), '| max weight:', round(w_iv.max(),4))

## 5. Define resilience metrics
Metrics are calculated on the held-out test period: annualized geometric return, annualized volatility, Sharpe ratio with zero risk-free rate, maximum drawdown, worst 5% daily conditional loss (CVaR), and longest underwater duration in trading days. Historical VaR/CVaR are empirical sample estimates, not guarantees about future loss.

In [ ]:
def perf_stats(r):
    r = pd.Series(r).dropna()
    wealth = (1+r).cumprod()
    dd = wealth/wealth.cummax()-1
    years = len(r)/252
    ann_return = wealth.iloc[-1]**(1/years)-1 if years > 0 and wealth.iloc[-1] > 0 else np.nan
    vol = r.std(ddof=1)*np.sqrt(252)
    sharpe = (r.mean()/r.std(ddof=1))*np.sqrt(252) if r.std(ddof=1)>0 else np.nan
    q05 = r.quantile(.05)
    cvar05 = r[r <= q05].mean()
    underwater = (dd < 0).to_numpy()
    run = longest = 0
    for flag in underwater:
        run = run + 1 if flag else 0
        longest = max(longest, run)
    return {'Annualized return':ann_return, 'Annualized volatility':vol, 'Sharpe (rf=0)':sharpe,
            'Max drawdown':dd.min(), 'VaR 5% daily':q05, 'CVaR 5% daily':cvar05,
            'Longest underwater (days)':longest, 'Test trading days':len(r)}
summary = pd.DataFrame({k:perf_stats(v) for k,v in test_returns.items()}).T
display(summary.style.format({'Annualized return':'{:.1%}','Annualized volatility':'{:.1%}','Sharpe (rf=0)':'{:.2f}','Max drawdown':'{:.1%}','VaR 5% daily':'{:.2%}','CVaR 5% daily':'{:.2%}'}))

## 6. Core result: return path and downside risk
The figure focuses on held-out cumulative wealth and drawdown. Compare portfolio lines on the same dates; do not infer that the best historical result is an optimal future allocation.

In [ ]:
wealth = pd.DataFrame({k:(1+v).cumprod() for k,v in test_returns.items()})
wealth.plot(lw=1.7, title='Held-out test period: growth of 1 unit (before costs)')
plt.ylabel('Growth of 1 unit'); plt.xlabel('Date'); plt.tight_layout()
plt.savefig(OUT/'02_test_wealth.png', dpi=170); plt.show()
drawdowns = wealth/wealth.cummax()-1
drawdowns.plot(lw=1.2, title='Held-out test-period drawdowns')
plt.ylabel('Drawdown'); plt.xlabel('Date'); plt.tight_layout()
plt.savefig(OUT/'03_test_drawdowns.png', dpi=170); plt.show()

## 7. Does diversification have diminishing marginal benefits?
For each breadth, we compare held-out volatility, CVaR, and drawdown. The breadth curve is a portfolio-construction diagnostic, not a universal law: all names are selected from the training-period liquid candidates, and the source universe may contain survivorship bias.

In [ ]:
breadth_rows = []
for n in breadths:
    r = test_returns[f'Equal weight {n}']
    s = perf_stats(r)
    breadth_rows.append({'Holdings':n,'Volatility':s['Annualized volatility'],'CVaR 5%':s['CVaR 5% daily'],'Max drawdown':s['Max drawdown']})
breadth_df = pd.DataFrame(breadth_rows).set_index('Holdings')
display(breadth_df.style.format('{:.2%}'))
fig, ax = plt.subplots()
breadth_df[['Volatility','Max drawdown']].abs().plot(marker='o', ax=ax)
ax.set(title='Absolute downside-risk measures by portfolio breadth', xlabel='Number of holdings', ylabel='Risk measure (absolute)')
ax.legend(['Annualized volatility','Maximum drawdown']); plt.tight_layout()
plt.savefig(OUT/'04_breadth_risk.png', dpi=170); plt.show()

## 8. Rolling-window stability
A full-period average can conceal regime dependence. We calculate rolling 252-trading-day metrics on the held-out period where data permits. If the test period is shorter than a year, this cell reports that rolling annual-window inference is unavailable rather than manufacturing a result.

In [ ]:
roll_name = f'Equal weight {min(20, max(breadths))}'
rr = test_returns[roll_name].sort_index()
if len(rr) >= 252:
    roll_vol = rr.rolling(252).std()*np.sqrt(252)
    roll_return = (1+rr).rolling(252).apply(np.prod, raw=True)**(252/252)-1
    fig, ax = plt.subplots()
    roll_vol.plot(ax=ax, label='Rolling annualized volatility', color='#d95f02')
    ax.set_ylabel('Annualized volatility'); ax.set_title(f'Rolling 252-day risk: {roll_name}')
    ax2 = ax.twinx(); roll_return.plot(ax=ax2, label='Rolling 252-day return', color='#1b9e77', alpha=.8)
    ax2.set_ylabel('Rolling 252-day return')
    ax.legend(loc='upper left'); ax2.legend(loc='upper right'); plt.tight_layout()
    plt.savefig(OUT/'05_rolling_stability.png', dpi=170); plt.show()
else:
    print(f'Only {len(rr)} test observations: insufficient for a 252-day rolling comparison. Extend the test history or report this limitation.')

## 9. Sensitivity: rebalance frequency and trading costs
Frequent turnover can erase apparent benefits. This section compares monthly, quarterly, and buy-and-hold equal-weight returns for the same top-20 training-selected holdings. Costs are illustrative basis-point deductions applied to estimated turnover at each rebalance; this is a simplified sensitivity, not a transaction-level execution model. The assumption is deliberately visible so readers can challenge it.

In [ ]:
def periodic_equal_weight_returns(panel, names, freq='M', cost_bps=0):
    p = panel[names].copy().sort_index()
    groups = p.index.to_period(freq)
    out = pd.Series(index=p.index, dtype=float)
    prev_w = pd.Series(0.0, index=names)
    for period in groups.unique():
        idx = groups == period; block = p.loc[idx]
        w = pd.Series(1/len(names), index=names)
        turnover = (w-prev_w).abs().sum()/2
        net = block.mul(w, axis=1).sum(axis=1)
        if len(net): net.iloc[0] -= turnover*cost_bps/10000
        out.loc[block.index] = net
        prev_w = w
    return out.dropna()
names20 = candidate_order[:min(20,len(candidate_order))]
cost_rows=[]
for freq in ['M','Q']:
    for bps in [0,5,15,30]:
        r=periodic_equal_weight_returns(test_panel,names20,freq=freq,cost_bps=bps)
        s=perf_stats(r)
        cost_rows.append({'Rebalance':freq,'Cost (bps/turnover)':bps,'Annualized return':s['Annualized return'],'Max drawdown':s['Max drawdown'],'CVaR 5%':s['CVaR 5% daily']})
cost_df=pd.DataFrame(cost_rows)
display(cost_df.style.format({'Annualized return':'{:.1%}','Max drawdown':'{:.1%}','CVaR 5%':'{:.2%}'}))
sns.lineplot(data=cost_df,x='Cost (bps/turnover)',y='Annualized return',hue='Rebalance',marker='o')
plt.title('Sensitivity of annualized return to assumed turnover costs'); plt.tight_layout()
plt.savefig(OUT/'06_cost_sensitivity.png',dpi=170); plt.show()

## 10. Concentration and correlation structure
Diversification only helps when holdings do not move identically. We inspect training-period return correlations for the 20 selected tickers and compare equal-weight portfolio volatility with the weighted average of constituent volatilities. This is a descriptive diversification diagnostic, not a causal estimate.

In [ ]:
corr=train_panel[names20].corr()
plt.figure(figsize=(12,9)); sns.heatmap(corr,cmap='vlag',center=0,vmin=-1,vmax=1,square=True)
plt.title('Training-period daily return correlation: selected 20 tickers'); plt.tight_layout()
plt.savefig(OUT/'07_training_correlations.png',dpi=170); plt.show()
constituent_vol=train_panel[names20].std()*np.sqrt(252)
ew_vol=train_panel[names20].mean(axis=1).std()*np.sqrt(252)
print('Equal-weight portfolio annualized volatility (train):',round(ew_vol,4))
print('Mean constituent annualized volatility (train):',round(constituent_vol.mean(),4))
print('Volatility reduction versus mean constituent:',round(1-ew_vol/constituent_vol.mean(),4))

## 11. Results: turn analysis into a decision memo
After running all cells, complete this section with values from `summary`, `breadth_df`, and `cost_df`. Do not claim diversification always works; state the period, portfolio definition, and cost assumption.

**Decision memo template**
- **Finding:** [Which portfolio had the smallest held-out drawdown/CVaR? Include value and dates.]
- **Trade-off:** [Did lower downside risk require sacrificing return? Quantify.]
- **Robustness:** [Did the breadth ranking persist across rolling windows and cost assumptions?]
- **Recommendation:** [A cautious portfolio-design implication supported by the analysis.]
- **What would change my mind:** [Name data needed next: adjusted prices, delisted constituents, benchmark/sector membership, transaction costs, longer history.]

### Conclusion boundary
The strongest defensible conclusion is about this file, this ticker screen, and this historical test period. It is not evidence of future returns or a recommendation to buy any security.

## 12. Limitations, reproducibility, and next research
- Historical sample ends in 2018; this is not current-market analysis.
- The source's split/dividend adjustment status is unverified. Results are price-return estimates until confirmed.
- A preselected or survivorship-biased ticker universe can omit failed/delisted companies and overstate results. A true point-in-time constituent history is needed to eliminate this.
- Liquidity screen uses training-period average dollar volume; missing observations and static common-panel selection can affect representativeness.
- Costs are simplified basis-point deductions; no tax, bid-ask spread, market impact, or actual fills are modeled.
- Historical CVaR and drawdown are sample-dependent and do not bound future losses. Sharpe assumes zero risk-free rate and 252 sessions per year.
- No fundamental or sector information is present, so this cannot explain business drivers or sector attribution.

**Reproducibility:** place the CSV beside this notebook, install `pandas`, `numpy`, `matplotlib`, and `seaborn`, then restart the kernel and run all cells. Generated figures are saved in `images/`. For Kaggle, attach the CSV as a notebook input and update `DATA_PATH`. Record source URL, license, and price-adjustment details in the published repository.